# Operator Unit Tests

| Test | Operator | What is checked | Expected result |
|------|----------|-----------------|------------------|
| G1   | **G**           | Gradient of linear $p$                                                                                        | Exact (machine $\varepsilon$)  |
| G2   | **G**           | Gradient of quadratic $p$                                                                                     | Error $O(\Delta x^2)$          |
| D1   | **DG**          | Composition $\approx \nabla^2$                                                                                | Error $O(\Delta x^2)$ interior |
| D2   | **D**           | Divergence of div-free field                                                                                  | Error $O(\Delta x^2)$ interior |
| D3   | **D**, **G**    | Adjoint: $\boldsymbol{u}^\top \mathbf{G} \boldsymbol{p} = \boldsymbol{p}^\top \mathbf{D} \boldsymbol{u}$    | Machine $\varepsilon$          |
| BC1  | $\mathbf{bc}_D$ | Global mass balance $\sum \mathbf{bc}_D = 0$                                                                  | Exact                          |
| BC2  | $\mathbf{bc}_L$ | Top-wall BC value $= 2U^*/\Delta y^2$                                                                         | Exact                          |
| L1   | **L**           | Laplacian of quadratic field $= 2$                                                                            | Exact interior                 |
| L2   | **L**           | Laplacian of linear field $= 0$                                                                               | Machine $\varepsilon$          |
| L3   | **L**           | Self-adjoint: $\boldsymbol{u}^\top \mathbf{L} \boldsymbol{v} = \boldsymbol{v}^\top \mathbf{L} \boldsymbol{u}$ | Machine $\varepsilon$         |
| A1   | **A**           | Uniform flow gives zero advection                                                                             | Machine $\varepsilon$          |
| A2   | **A**           | Energy: $\boldsymbol{u}^\top \mathbf{A}(\boldsymbol{u}) \approx 0$                                           | Small $(O(\Delta x^2))$        |
| CG1  | CG              | Solves Poisson problem to tolerance                                                                           | $< \text{tol}$                 |
| FULL | **T**           | $\boldsymbol{p}^\top \mathbf{T} \boldsymbol{p} > 0$ (SPD)                                                    | Positive                       |

## Imports

In [1]:
import numpy as np
import operators as ops

## Grid Setup

Shared across all tests. Uses a small grid (8×8) so tests run fast.

In [2]:
nx, ny = 8, 8        # small grid for fast unit tests
Lx, Ly = 1.0, 1.0
dx, dy = Lx / nx, Ly / ny
delta  = dx           # dx == dy for this square uniform grid

n_cells = nx * ny
n_vel   = ny * (nx - 1) + nx * (ny - 1)

# Index pointer arrays (same logic as func_gen_pointer in project_2.ipynb)
ip = np.full((nx, ny),     -1, dtype=int)
iu = np.full((nx + 1, ny), -1, dtype=int)
iv = np.full((nx, ny + 1), -1, dtype=int)

count = 0
for i in range(nx):
    for j in range(ny):
        ip[i, j] = count; count += 1

count = 0
for i in range(1, nx):       # interior vertical faces
    for j in range(ny):
        iu[i, j] = count; count += 1

count = ny * (nx - 1)  # v-DOFs follow u-DOFs in the flat velocity vector
for i in range(nx):
    for j in range(1, ny):   # interior horizontal faces
        iv[i, j] = count; count += 1

# Physical coordinates at pressure cell centers
xp = np.linspace(dx / 2, Lx - dx / 2, nx)
yp = np.linspace(dy / 2, Ly - dy / 2, ny)
XI, YJ = np.meshgrid(xp, yp, indexing='ij')   # shape (nx, ny)

## G1

In [3]:
# G1: gradient of linear pressure p = a*x + b*y
# For a linear field, finite differences are exact -> error should be machine epsilon.
# Expected: every u-face entry == a, every v-face entry == b

a, b = 3.0, 5.0

p = np.zeros(n_cells)
p[ip] = a * XI + b * YJ          # p(x,y) = a*x + b*y at each cell center

grad = ops.G(p, delta, ip, iu, iv, nx, ny, n_vel)

err_u = np.max(np.abs(grad[iu[1:nx, :ny]] - a))   # u-faces should all equal a
err_v = np.max(np.abs(grad[iv[:nx, 1:ny]] - b))   # v-faces should all equal b

print(f"G1  u-error: {err_u:.2e}  (expect ~machine ε ≈ 2e-16)")
print(f"G1  v-error: {err_v:.2e}  (expect ~machine ε ≈ 2e-16)")
assert err_u < 1e-12 and err_v < 1e-12, "G1 FAILED"
print("G1 PASSED")

G1  u-error: 0.00e+00  (expect ~machine ε ≈ 2e-16)
G1  v-error: 0.00e+00  (expect ~machine ε ≈ 2e-16)
G1 PASSED


## G2

In [4]:
# G2: gradient of quadratic pressure p = x^2 + y^2
# The staggered-grid finite difference is exact for quadratics

p = np.zeros(n_cells)
p[ip] = XI**2 + YJ**2

grad = ops.G(p, delta, ip, iu, iv, nx, ny, n_vel)

# Physical coordinates of the velocity faces
xu_faces = np.linspace(dx, (nx - 1) * dx, nx - 1)   # u-face x-coords, i=1..nx-1
yv_faces = np.linspace(dy, (ny - 1) * dy, ny - 1)   # v-face y-coords, j=1..ny-1

# Exact gradient: dp/dx = 2x at each u-face, dp/dy = 2y at each v-face
XU, _  = np.meshgrid(xu_faces, yp,       indexing='ij')   # shape (nx-1, ny)
_,  YV = np.meshgrid(xp,       yv_faces, indexing='ij')   # shape (nx,   ny-1)

err_u = np.max(np.abs(grad[iu[1:nx, :ny]]  - 2 * XU))
err_v = np.max(np.abs(grad[iv[:nx, 1:ny]]  - 2 * YV))

print(f"G2  u-error: {err_u:.2e}  (expect ~machine ε)")
print(f"G2  v-error: {err_v:.2e}  (expect ~machine ε)")
assert err_u < 1e-12 and err_v < 1e-12, "G2 FAILED"
print("G2 PASSED")

G2  u-error: 0.00e+00  (expect ~machine ε)
G2  v-error: 0.00e+00  (expect ~machine ε)
G2 PASSED


## D1

In [5]:
# D1: D(G(p)) should approximate the Laplacian of p for interior cells
# p = x^2 + y^2  ->  nabla^2 p = 4 everywhere
# Expected: machine epsilon (staggered FD is exact for quadratics)

p_test = np.zeros(n_cells)
p_test[ip] = XI**2 + YJ**2

grad = ops.G(p_test, delta, ip, iu, iv, nx, ny, n_vel)
lap  = ops.D(grad,   delta, ip, iu, iv, nx, ny, n_cells)

exact = 4.0
lap_interior = lap[ip[1:nx-1, 1:ny-1]]   # interior cells only
err = np.max(np.abs(lap_interior - exact))

print(f"D1  max error (interior): {err:.2e}  (expect ~machine eps)")
assert err < 1e-10, "D1 FAILED"
print("D1 PASSED")

D1  max error (interior): 0.00e+00  (expect ~machine eps)
D1 PASSED


## D2

In [6]:
# D2: D applied to a div-free velocity field should give ~0 (O(dx^2) error)
# Analytical field: u = sin(pi*x)cos(pi*y), v = -cos(pi*x)sin(pi*y)
# du/dx + dv/dy = pi*cos(pi*x)cos(pi*y) - pi*cos(pi*x)cos(pi*y) = 0

# u-face coordinates: x = i*dx (i=1..nx-1), y = (j+0.5)*dy (j=0..ny-1)
xu_f = np.arange(1, nx) * delta
yp_c = (np.arange(ny) + 0.5) * delta
XU_f, YU_f = np.meshgrid(xu_f, yp_c, indexing='ij')   # (nx-1, ny)

# v-face coordinates: x = (i+0.5)*dx (i=0..nx-1), y = j*dy (j=1..ny-1)
xp_c = (np.arange(nx) + 0.5) * delta
yv_f = np.arange(1, ny) * delta
XV_f, YV_f = np.meshgrid(xp_c, yv_f, indexing='ij')   # (nx, ny-1)

vel_df = np.zeros(n_vel)
vel_df[iu[1:nx, :ny]] =  np.sin(np.pi * XU_f) * np.cos(np.pi * YU_f)
vel_df[iv[:nx, 1:ny]] = -np.cos(np.pi * XV_f) * np.sin(np.pi * YV_f)

div_df = ops.D(vel_df, delta, ip, iu, iv, nx, ny, n_cells)

# Interior cells only (boundary cells missing wall-face contributions from bc_D)
err = np.max(np.abs(div_df[ip[1:nx-1, 1:ny-1]]))

print(f"D2  max |div| (interior): {err:.2e}  (expect O(dx^2) ~ {delta**2:.2e})")
assert err < delta**2 * 10, "D2 FAILED"
print("D2 PASSED")

D2  max |div| (interior): 8.88e-16  (expect O(dx^2) ~ 1.56e-02)
D2 PASSED


## D3

In [7]:
# D3: Adjoint relationship  u^T (G p) = -p^T (D u)  (machine eps with homogeneous BCs)
# This is the discrete integration-by-parts identity:
#   integral u . grad(p) dV = -integral p div(u) dV  (zero-BC surface term drops out)
# Equivalently: u^T G p + p^T D u = 0
# Note: the PDF table writes '= p^T D u' but that omits the minus sign from IBP.

rng    = np.random.default_rng(42)
u_rand = rng.standard_normal(n_vel)
p_rand = rng.standard_normal(n_cells)

lhs = u_rand @ ops.G(p_rand, delta, ip, iu, iv, nx, ny, n_vel)
rhs = p_rand @ ops.D(u_rand, delta, ip, iu, iv, nx, ny, n_cells)

# They should be exact negatives: lhs + rhs = 0
err = abs(lhs + rhs) / (abs(lhs) + 1e-300)

print(f"D3  |u^T G p + p^T D u| / |lhs|: {err:.2e}  (expect ~machine eps ~ 1e-16)")
assert err < 1e-12, "D3 FAILED"
print("D3 PASSED")

D3  |u^T G p + p^T D u| / |lhs|: 0.00e+00  (expect ~machine eps ~ 1e-16)
D3 PASSED
